# Spain's economy by prime minister, step by step

A Spanish news outlet published an Instagram carousel comparing Pedro Sánchez (2018 to 2026) with his three predecessors, Aznar, Zapatero and Rajoy, on a dozen economic indicators. Its headline claims:

1. GDP grew **15.1%** under Sánchez, well above Rajoy (10.3%) and Zapatero (7.1%).
2. The minimum wage (SMI) rose about **66%**, far ahead of inflation.
3. Employment rose **18 to 19%** and unemployment fell to its lowest since 2008.
4. Housing is the weak spot: price per m² up **48.3%**, versus a **6.7%** fall under Rajoy.

In this notebook we:

* **Source** the underlying data ourselves from Eurostat.
* **Reproduce** the article's comparison table.
* **Stress-test** it four ways: annualizing, going per capita, separating nominal from real, and benchmarking against the euro area.

This is the step-by-step version of `spain_economy_by_term.ipynb`. It reaches the same results, but each cell does one small thing so you can run it, look at the output, and then move on.

The goal is not to decide who was the better prime minister. It is to see how much a chart's *framing choices* shape the story told by the same numbers.

## 0. Setup

We use two libraries: **polars** for tables and **matplotlib** for charts. `urllib.request` is part of standard Python and downloads files from the web.

In [ ]:
import urllib.request

import matplotlib.pyplot as plt
import polars as pl

pl.Config.set_tbl_rows(40)  # show up to 40 rows when we print a table

## 1. Sourcing the data

Everything comes from [Eurostat](https://ec.europa.eu/eurostat/web/main/data/database), which republishes harmonized national statistics from Spain's INE, Banco de España and Social Security. Using one source keeps definitions consistent across 30 years and gives us the euro area as a comparison group.

| Indicator | Eurostat dataset | Notes |
|---|---|---|
| Real GDP | `nama_10_gdp` | Chain-linked volumes, 2010 euros |
| Real GDP per capita | `nama_10_pc` | Chain-linked, euros per inhabitant |
| Prices (inflation) | `prc_hicp_aind` | HICP annual average index |
| Minimum wage | `earn_mw_cur` | Monthly, euros, semiannual; from 1999 |
| House prices | `prc_hpi_a` | House price index; Spain from 2007 |
| Employment | `nama_10_pe` | Persons employed, national accounts |
| Unemployment rate | `une_rt_a` | Age 15 to 74, % of labour force |
| Public debt | `gov_10dd_edpt1` | General government, % of GDP |
| Population | `demo_pjan` | 1 January |

**Differences from the article.** The article uses Spanish sources directly: the EPA survey and Social Security affiliates for jobs, the ministry's €/m² valuation series for housing, and the legal SMI table. Our versions are close cousins, not identical, so expect our numbers to differ by a few points. Two gaps matter most: Eurostat's minimum wage series starts in 1999 (so Aznar's term is partial), and its house price index for Spain starts in 2007 (so Aznar has no housing figure).

### Step 1.1: Download one dataset by hand

Eurostat can send any dataset as a CSV file. The web address (URL) has three parts:

1. the base address of the Eurostat service,
2. the dataset name, here `nama_10_gdp`,
3. filters that pick the rows we want. `c[geo]=ES,EA20` means "Spain and the euro area", `c[unit]=CLV10_MEUR` means "millions of 2010 euros" and `c[na_item]=B1GQ` means "GDP".

In [ ]:
BASE = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/"
FORMAT = "&format=csvdata&formatVersion=2.0&compress=false"

url = BASE + "nama_10_gdp/1.0?" + "c[geo]=ES,EA20&c[unit]=CLV10_MEUR&c[na_item]=B1GQ" + FORMAT
print(url)

Now we download the file and read it into a polars table (a *DataFrame*).

In [ ]:
csv_bytes = urllib.request.urlopen(url, timeout=120).read()
gdp_raw = pl.read_csv(csv_bytes)
gdp_raw.head()

Most columns repeat the filters we asked for. The three we need are:

* `geo`: the country (`ES` = Spain, `EA20` = euro area),
* `TIME_PERIOD`: the year,
* `OBS_VALUE`: the number itself.

In [ ]:
gdp_raw = gdp_raw.select("geo", "TIME_PERIOD", "OBS_VALUE")
gdp_raw.head()

### Step 1.2: Keep Spain only, with simple column names

`filter` keeps the rows where a condition is true. `rename` gives the columns names that are easier to type.

In [ ]:
gdp = gdp_raw.filter(pl.col("geo") == "ES")
gdp = gdp.rename({"TIME_PERIOD": "year", "OBS_VALUE": "gdp"})
gdp = gdp.select("year", "gdp")
gdp.head()

**Try it:** make the same table for the euro area. Change `"ES"` to `"EA20"` and call the column `gdp_ea`.

In [ ]:
gdp_ea = gdp_raw.filter(pl.col("geo") == "EA20")
gdp_ea = gdp_ea.rename({"TIME_PERIOD": "year", "OBS_VALUE": "gdp_ea"})
gdp_ea = gdp_ea.select("year", "gdp_ea")
gdp_ea.head()

### Step 1.3: Two small helper functions

We have eight more datasets to download, and each one needs the same steps. Rather than copy them eight times, we put them in two short functions:

* `download(dataset, filters)` builds the URL and returns the table.
* `keep_year_value(table, name, geo)` keeps one country and renames the columns.

In [ ]:
def download(dataset, filters):
    url = BASE + dataset + "/1.0?" + filters + FORMAT
    csv_bytes = urllib.request.urlopen(url, timeout=120).read()
    return pl.read_csv(csv_bytes)


def keep_year_value(table, name, geo="ES"):
    table = table.filter(pl.col("geo") == geo)
    table = table.rename({"TIME_PERIOD": "year", "OBS_VALUE": name})
    return table.select("year", name)

### Step 1.4: Download the rest, one at a time

Each cell downloads one dataset and prints how many rows it got. If Eurostat ever renames a code, only that cell fails, and you only need to fix that one line.

In [ ]:
gdp_pc_raw = download("nama_10_pc", "c[geo]=ES,EA20&c[unit]=CLV10_EUR_HAB&c[na_item]=B1GQ")
gdp_pc = keep_year_value(gdp_pc_raw, "gdp_pc")
gdp_pc_ea = keep_year_value(gdp_pc_raw, "gdp_pc_ea", geo="EA20")
print("GDP per capita:", gdp_pc.height, "rows for Spain,", gdp_pc_ea.height, "for the euro area")

In [ ]:
hicp_raw = download("prc_hicp_aind", "c[geo]=ES&c[unit]=INX_A_AVG&c[coicop]=CP00")
hicp = keep_year_value(hicp_raw, "hicp")
print("Prices (HICP):", hicp.height, "rows")

In [ ]:
hpi_raw = download("prc_hpi_a", "c[geo]=ES&c[unit]=I15_A_AVG&c[purchase]=TOTAL")
hpi = keep_year_value(hpi_raw, "hpi")
print("House prices:", hpi.height, "rows")

In [ ]:
emp_raw = download("nama_10_pe", "c[geo]=ES&c[unit]=THS_PER&c[na_item]=EMP_DC")
emp = keep_year_value(emp_raw, "emp")
print("Employment:", emp.height, "rows")

In [ ]:
unemp_raw = download("une_rt_a", "c[geo]=ES&c[age]=Y15-74&c[sex]=T&c[unit]=PC_ACT")
unemp = keep_year_value(unemp_raw, "unemp")
print("Unemployment rate:", unemp.height, "rows")

In [ ]:
debt_raw = download("gov_10dd_edpt1", "c[geo]=ES&c[sector]=S13&c[na_item]=GD&c[unit]=PC_GDP")
debt = keep_year_value(debt_raw, "debt")
print("Public debt:", debt.height, "rows")

In [ ]:
pop_raw = download("demo_pjan", "c[geo]=ES&c[age]=TOTAL&c[sex]=T&c[unit]=NR")
pop = keep_year_value(pop_raw, "pop")
print("Population:", pop.height, "rows")

### Step 1.5: The minimum wage needs one extra step

The minimum wage is published twice a year, so its periods look like `2024-S1` and `2024-S2` instead of `2024`. Let's look:

In [ ]:
smi_raw = download("earn_mw_cur", "c[geo]=ES&c[currency]=EUR")
smi_raw.select("TIME_PERIOD", "OBS_VALUE").head()

Spain sets the minimum wage in January, so we keep the first half of each year (`S1`). Then we turn `"2024-S1"` into the number `2024` by taking its first four characters.

In [ ]:
smi = smi_raw.filter(pl.col("TIME_PERIOD").str.ends_with("S1"))
smi = smi.with_columns(pl.col("TIME_PERIOD").str.slice(0, 4).cast(pl.Int64))
smi = keep_year_value(smi, "smi")
smi.head()

### Step 1.6: Join everything into one table

We start with a table that has one row per year, 1995 to 2026, and add each series as a new column. A *left join* keeps every year, and leaves a blank (`null`) where a series has no value for that year.

In [ ]:
es = pl.DataFrame({"year": range(1995, 2027)})

es = es.join(gdp, on="year", how="left")
es = es.join(gdp_ea, on="year", how="left")
es = es.join(gdp_pc, on="year", how="left")
es = es.join(gdp_pc_ea, on="year", how="left")
es = es.join(hicp, on="year", how="left")
es = es.join(smi, on="year", how="left")
es = es.join(hpi, on="year", how="left")
es = es.join(emp, on="year", how="left")
es = es.join(unemp, on="year", how="left")
es = es.join(debt, on="year", how="left")
es = es.join(pop, on="year", how="left")
es

### Step 1.7: Add inflation-adjusted (real) columns

To remove inflation from a series, divide it by the price index. `with_columns` adds a new column computed from existing ones.

In [ ]:
es = es.with_columns((pl.col("smi") / pl.col("hicp") * 100).alias("smi_real"))
es = es.with_columns((pl.col("hpi") / pl.col("hicp") * 100).alias("hpi_real"))
es.select("year", "smi", "smi_real", "hpi", "hpi_real").tail(8)

### Step 1.8: Define the terms

We approximate each term with calendar years, as the article's annual charts do. Sánchez took office in June 2018, so 2018 counts as both Rajoy's end year and Sánchez's start year. Sánchez's end year is the latest year with GDP data.

In [ ]:
LAST = es.drop_nulls("gdp")["year"].max()
print("Latest full year of GDP data:", LAST)

In [ ]:
TERMS = [
    ("Aznar (PP)",      1996, 2004),
    ("Zapatero (PSOE)", 2004, 2011),
    ("Rajoy (PP)",      2011, 2018),
    ("Sánchez (PSOE)",  2018, LAST),
]

for term, start, end in TERMS:
    print(term, "from", start, "to", end, "=", end - start, "years")

## 2. Reproducing the article's table

First, the article's own numbers, transcribed from the graphic (using the 2018 to 2026 column where it exists).

### Original Instagram graphics

The comparison table and accompanying charts are included below for reference. The images are loaded from GitHub so they also appear in Colab; the `ig-imagenes` folder must be published on the repository's `main` branch.

<table>
  <tr>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/Colores.jpeg" alt="Original comparison table" width="360"><br>Comparison table</td>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/PIB.jpeg" alt="Minimum wage and inflation chart" width="360"><br>Minimum wage and inflation</td>
  </tr>
  <tr>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/PIB_cuartos.jpeg" alt="GDP growth chart" width="360"><br>GDP growth</td>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/Vivienda.jpeg" alt="Housing prices and wages chart" width="360"><br>Housing</td>
  </tr>
  <tr>
    <td align="center"><img src="https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/Economia/ig-imagenes/migracion.jpeg" alt="Employment chart" width="360"><br>Employment</td>
    <td></td>
  </tr>
</table>

In [ ]:
article = pl.DataFrame(
    {
        "indicator": ["Real GDP", "Real GDP per capita", "Inflation", "Minimum wage (nominal)",
                      "Minimum wage (real)", "House prices (nominal)", "House prices (real)",
                      "Employment", "Unemployment rate (pp)", "Public debt (pp of GDP)"],
        "Aznar (PP)":      [34.5, 25.2, 25.1, 18.0, -5.7, 120.5, 76.3, 41.3, -9.0, -20.1],
        "Zapatero (PSOE)": [7.1, -1.3, 19.9, 39.3, 16.2, 10.6, -7.8, 1.5, 10.4, 24.2],
        "Rajoy (PP)":      [10.3, 9.7, 6.9, 14.7, 7.3, -6.7, -12.7, 4.9, -6.1, 30.3],
        "Sánchez (PSOE)":  [15.1, 6.2, 26.5, 65.9, 31.2, 48.3, 17.2, 17.9, -5.4, 1.8],
    }
)
article

### Step 2.1: One number by hand

How much did real GDP grow under Sánchez? Take the value in the first year and the value in the last year:

$$\text{\% change} = \left(\frac{\text{end value}}{\text{start value}} - 1\right) \times 100$$

In [ ]:
start_value = es.filter(pl.col("year") == 2018)["gdp"][0]
end_value = es.filter(pl.col("year") == LAST)["gdp"][0]
print("GDP in 2018:", start_value)
print("GDP in", LAST, ":", end_value)

In [ ]:
growth = (end_value / start_value - 1) * 100
print(f"Real GDP growth under Sánchez: {growth:.1f}%")

Compare with the article's 15.1%.

### Step 2.2: Turn the steps into functions

We will repeat this calculation many times, so we wrap each step in a short function.

* `get_value(column, year)` looks up one number.
* `pct_change(column, start, end)` is the percent change between two years.
* `pp_change(column, start, end)` is the change in percentage points. We use it for things that are already percentages, like the unemployment rate: going from 15% to 10% is a change of −5 points.

Both change functions return `None` (no answer) when a year is missing.

In [ ]:
def get_value(column, year):
    return es.filter(pl.col("year") == year)[column][0]


def pct_change(column, start, end):
    a = get_value(column, start)
    b = get_value(column, end)
    if a is None or b is None:
        return None
    return (b / a - 1) * 100


def pp_change(column, start, end):
    a = get_value(column, start)
    b = get_value(column, end)
    if a is None or b is None:
        return None
    return b - a

Check that the function gives the same answer as our hand calculation:

In [ ]:
pct_change("gdp", 2018, LAST)

### Step 2.3: Build our version of the table

The list below says, for each row of the table, which column of `es` to use and whether it is a percent change (`"pct"`) or a change in points (`"pp"`). Then two loops fill in the table: the outer loop goes over indicators, the inner loop over terms.

In [ ]:
INDICATORS = [
    # (label in the table, column in es, kind)
    ("Real GDP", "gdp", "pct"),
    ("Real GDP per capita", "gdp_pc", "pct"),
    ("Inflation", "hicp", "pct"),
    ("Minimum wage (nominal)", "smi", "pct"),
    ("Minimum wage (real)", "smi_real", "pct"),
    ("House prices (nominal)", "hpi", "pct"),
    ("House prices (real)", "hpi_real", "pct"),
    ("Employment", "emp", "pct"),
    ("Unemployment rate (pp)", "unemp", "pp"),
    ("Public debt (pp of GDP)", "debt", "pp"),
    ("Population", "pop", "pct"),
]

In [ ]:
rows = []
for label, column, kind in INDICATORS:
    row = {"indicator": label}
    for term, start, end in TERMS:
        if kind == "pct":
            row[term] = pct_change(column, start, end)
        else:
            row[term] = pp_change(column, start, end)
    rows.append(row)

ours = pl.DataFrame(rows)
ours = ours.with_columns(pl.exclude("indicator").round(1))
ours

### Step 2.4: Compare with the article, one term at a time

We take the article's column and our column for one term and put them next to each other. Change `term` to look at another prime minister.

In [ ]:
term = "Sánchez (PSOE)"

article_col = article.select("indicator", pl.col(term).alias("article"))
ours_col = ours.select("indicator", pl.col(term).alias("ours"))

compare = article_col.join(ours_col, on="indicator")
compare = compare.with_columns((pl.col("ours") - pl.col("article")).round(1).alias("difference"))
compare

**Check:** where do we match closely, and where do differences in source (EPA vs national accounts employment, €/m² valuations vs a price index) or missing years show up?

## 3. Reproducing the article's charts

Each chart marks the start of every term with a dashed line and the prime minister's name.

### Step 3.1: Annual real GDP growth (article slide 5)

`pct_change()` on a column gives the change from the previous row, so here it gives growth from one year to the next.

In [ ]:
growth = es.select("year", (pl.col("gdp").pct_change() * 100).alias("growth"))
growth = growth.drop_nulls()
growth.head()

We colour good years blue and bad years red.

In [ ]:
bar_colors = []
for g in growth["growth"]:
    if g >= 0:
        bar_colors.append("#2c5aa0")
    else:
        bar_colors.append("#c0392b")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(growth["year"], growth["growth"], color=bar_colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Real GDP growth, % y/y")
ax.set_title("How GDP did in each term")

top = ax.get_ylim()[1]
for term, start, end in TERMS:
    ax.axvline(start, color="gray", linestyle="--", linewidth=0.8)
    ax.text(start + 0.2, top, term, va="top", fontsize=9, style="italic")

plt.show()

### Step 3.2: Minimum wage vs prices (article slide 3)

To compare two series measured in different units (euros and an index), we set both to 100 in the same starting year. The minimum wage data begins in 1999, so we start there.

In [ ]:
base_year = es.drop_nulls("smi")["year"].min()
smi_base = get_value("smi", base_year)
hicp_base = get_value("hicp", base_year)
print("Base year:", base_year, "| minimum wage:", smi_base, "| price index:", hicp_base)

In [ ]:
index = es.filter(pl.col("year") >= base_year)
index = index.with_columns((pl.col("smi") / smi_base * 100).alias("smi_index"))
index = index.with_columns((pl.col("hicp") / hicp_base * 100).alias("hicp_index"))
index.select("year", "smi_index", "hicp_index").head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(index["year"], index["smi_index"], marker="o", markersize=3, color="#2c5aa0", label="Minimum wage")
ax.plot(index["year"], index["hicp_index"], marker="o", markersize=3, color="#c0392b", label="Prices (HICP)")
ax.set_title(f"Minimum wage vs inflation, {base_year} = 100")
ax.legend()

top = ax.get_ylim()[1]
for term, start, end in TERMS:
    ax.axvline(start, color="gray", linestyle="--", linewidth=0.8)
    ax.text(start + 0.2, top, term, va="top", fontsize=9, style="italic")

plt.show()

### Step 3.3: Employment (article slide 4)

Employment is in thousands of people; dividing by 1000 gives millions.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(es["year"], es["emp"] / 1000, color="#8fb0de")
ax.set_ylim(0, None)
ax.set_ylabel("Employed persons, millions")
ax.set_title("Employment (national accounts)")

top = ax.get_ylim()[1]
for term, start, end in TERMS:
    ax.axvline(start, color="gray", linestyle="--", linewidth=0.8)
    ax.text(start + 0.2, top, term, va="top", fontsize=9, style="italic")

plt.show()

### Step 3.4: Is unemployment the lowest since 2008?

We look up the latest unemployment rate, then search the earlier years for one that was as low or lower.

In [ ]:
latest_unemp = get_value("unemp", LAST)
print(f"Unemployment in {LAST}: {latest_unemp:.1f}%")

earlier = es.filter(pl.col("year") < LAST)
as_low = earlier.filter(pl.col("unemp") <= latest_unemp)
print("Earlier years at or below that level:", as_low["year"].to_list())

first_year = es.drop_nulls("unemp")["year"].min()
print("Our unemployment data starts in:", first_year)

**Check the claim.** The article says unemployment is at its lowest since 2008. An empty list means no year in our data was as low, but notice when our data starts: can our data confirm "since 2008"? Also notice what "lowest since 2008" leaves unsaid: how high is the rate in absolute terms? We come back to this with the euro-area comparison.

## 4. Stress test 1: annualize

Terms have different lengths. A cumulative percentage rewards a longer term even at the same pace. Converting to a compound annual rate puts every term on the same clock:

$$\text{annual rate} = \left(\frac{x_\text{end}}{x_\text{start}}\right)^{1/\text{years}} - 1$$

### Step 4.1: One annual rate by hand

In [ ]:
years = LAST - 2018
total_ratio = get_value("gdp", LAST) / get_value("gdp", 2018)
annual = (total_ratio ** (1 / years) - 1) * 100
print(f"Sánchez: {years} years, total growth {(total_ratio - 1) * 100:.1f}%, or {annual:.2f}% per year")

### Step 4.2: Functions and the annualized table

For rates measured in points (unemployment, debt) we divide the change by the number of years.

In [ ]:
def annual_pct(column, start, end):
    a = get_value(column, start)
    b = get_value(column, end)
    if a is None or b is None:
        return None
    years = end - start
    return ((b / a) ** (1 / years) - 1) * 100


def annual_pp(column, start, end):
    a = get_value(column, start)
    b = get_value(column, end)
    if a is None or b is None:
        return None
    years = end - start
    return (b - a) / years

The loop is the same as in Step 2.3, with the annual functions instead.

In [ ]:
rows = []
for label, column, kind in INDICATORS:
    row = {"indicator": label}
    for term, start, end in TERMS:
        if kind == "pct":
            row[term] = annual_pct(column, start, end)
        else:
            row[term] = annual_pp(column, start, end)
    rows.append(row)

annual_table = pl.DataFrame(rows)
annual_table = annual_table.with_columns(pl.exclude("indicator").round(1))
annual_table

**Question.** The GDP slide compares Sánchez with Rajoy and Zapatero but not Aznar. Using the annualized table, where does each term rank on real GDP growth? Does the ranking change compared with the cumulative table?

## 5. Stress test 2: go per capita

Spain's population grew substantially over this period, much of it through immigration. More people means more output even if the average person is no better off. Real GDP *per capita* is the better measure of living standards.

### Step 5.1: Pick out the three rows we need

In [ ]:
keep = ["Real GDP", "Real GDP per capita", "Population"]
annual_table.filter(pl.col("indicator").is_in(keep))

### Step 5.2: Total vs per person, as a chart

First we make two lists of numbers, one value per term.

In [ ]:
names = []
total_growth = []
per_person_growth = []
for term, start, end in TERMS:
    names.append(term)
    total_growth.append(annual_pct("gdp", start, end))
    per_person_growth.append(annual_pct("gdp_pc", start, end))

print(names)
print(total_growth)
print(per_person_growth)

To draw two bars side by side for each term, we place the first bar a little to the left of each position and the second a little to the right.

In [ ]:
positions = range(len(names))
left = [p - 0.2 for p in positions]
right = [p + 0.2 for p in positions]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(left, total_growth, width=0.4, label="Total real GDP", color="#2c5aa0")
ax.bar(right, per_person_growth, width=0.4, label="Real GDP per capita", color="#8fb0de")
ax.set_xticks(list(positions), names)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("% per year")
ax.set_title("Annualized growth: total vs per person")
ax.legend()
plt.show()

**Question.** How much of each term's GDP growth came from more people rather than more output per person? Which term's ranking moves the most when you switch measures? Notice that the article's own table contains the per capita row, but the GDP headline uses the total.

## 6. Stress test 3: nominal vs real

The minimum wage and housing headlines use nominal figures (66%, 48.3%), while the real figures sit in the table. Over a term with cumulative inflation in the 20s, nominal and real tell quite different stories.

In [ ]:
keep = ["Inflation", "Minimum wage (nominal)", "Minimum wage (real)",
        "House prices (nominal)", "House prices (real)"]
ours.filter(pl.col("indicator").is_in(keep))

**Question.** The housing slide contrasts +48.3% under Sánchez with −6.7% under Rajoy. Rajoy's term began in 2011, just after the housing bubble burst. How much of each figure is the policy of the government in office and how much is the phase of the housing cycle it inherited? What would a fair comparison look like?

## 7. Stress test 4: benchmark against the euro area

Every term coincided with a shared shock: the euro boom (Aznar), the Great Recession (Zapatero), the euro debt crisis and recovery (Rajoy), COVID and the energy shock (Sánchez). One way to separate a government's performance from the global weather is to compare Spain with its peers over the same years.

### Step 7.1: Spain and the euro area, per person, per year

In [ ]:
spain = []
euro_area = []
for term, start, end in TERMS:
    spain.append(annual_pct("gdp_pc", start, end))
    euro_area.append(annual_pct("gdp_pc_ea", start, end))

bench = pl.DataFrame({"term": names, "Spain": spain, "Euro area": euro_area})
bench = bench.with_columns((pl.col("Spain") - pl.col("Euro area")).alias("Spain minus EA"))
bench = bench.with_columns(pl.exclude("term").round(2))
bench

### Step 7.2: Chart the gap

Blue means Spain grew faster than the euro area, red means slower.

In [ ]:
gap_colors = []
for g in bench["Spain minus EA"]:
    if g >= 0:
        gap_colors.append("#2c5aa0")
    else:
        gap_colors.append("#c0392b")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(bench["term"], bench["Spain minus EA"], color=gap_colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("percentage points per year")
ax.set_title("Spain's real GDP per capita growth relative to the euro area")
plt.show()

**Caveat.** The euro area series uses today's 20-member composition for every year, and Spain is itself part of it (roughly a tenth of euro-area GDP), which mutes the gap slightly.

## 8. What the slides leave out

The carousel turns four indicators into headline slides. Two rows of its own table get no slide:

In [ ]:
keep = ["Inflation", "Public debt (pp of GDP)"]
ours.filter(pl.col("indicator").is_in(keep))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(es["year"], es["debt"], color="#2c5aa0")
ax.set_title("Public debt, % of GDP")

top = ax.get_ylim()[1]
for term, start, end in TERMS:
    ax.axvline(start, color="gray", linestyle="--", linewidth=0.8)
    ax.text(start + 0.2, top, term, va="top", fontsize=9, style="italic")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(es["year"], es["unemp"], color="#c0392b")
ax.set_title("Unemployment rate, %")

top = ax.get_ylim()[1]
for term, start, end in TERMS:
    ax.axvline(start, color="gray", linestyle="--", linewidth=0.8)
    ax.text(start + 0.2, top, term, va="top", fontsize=9, style="italic")

plt.show()

A change of +1.8 points of GDP in debt sounds like stability, and given COVID spending it arguably is. But the *level* it stabilized at matters for the next government. The same goes for unemployment: a multi-year low can still be high relative to peers.

## 9. Discussion

1. Rank the four terms on real GDP growth three ways: cumulative, annualized, and annualized per capita. Which framing would each party's press office choose?
2. The article picked calendar endpoints. Re-run the table with Sánchez starting in 2019 (his first full year). How sensitive are the results to a one-year shift? Edit `TERMS` in Step 1.8 and re-run the notebook.
3. Pick one headline from the carousel and rewrite it so that it is equally short but harder to mislead with.
4. Which single indicator do you think best captures "how people's lives went" during a term? What does it miss?